In [ ]:
# this script is for using checkpointers to implement fault 
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env file
from langgraph.checkpoint.memory import InMemorySaver
import time

In [ ]:

class State(TypedDict):
    input: str
    step1_output: str
    step2_output: str
    step3_output: str

In [ ]:
def step1(state: State):
    print("performed step 1")
    return {'step1_output': "done"}

def step2(state: State):
    print("Executing step 2. Hanged. Simulating failure...")
    time.sleep(30)  # Simulate a hang or failure
    return {'step2_output': "done"}

def step3(state: State):
    print("performed step 3")
    return {'step3_output': "done"}

In [ ]:
checkpointer = InMemorySaver()

graph = StateGraph(State)

graph.add_node("step1", step1)
graph.add_node("step2", step2)
graph.add_node("step3", step3)

graph.add_edge(START, "step1")
graph.add_edge("step1", "step2")
graph.add_edge("step2", "step3")
graph.add_edge("step3", END)

workflow = graph.compile(checkpointer=checkpointer)

workflow

In [10]:
thread_id = 42
config = {'configurable':{'thread_id': thread_id}}

initial_state = {'input': 'start'}


try:
    print("executing workflow. You can interrupt (Ctrl+C) to simulate failure.")
    final_state = workflow.invoke(initial_state, config=config)
except KeyboardInterrupt:
    print("Workflow interrupted. You can resume later.")

final_state

executing workflow. You can interrupt (Ctrl+C) to simulate failure.
performed step 1
Executing step 2. Hanged. Simulating failure...
performed step 3


{'input': 'start',
 'step1_output': 'done',
 'step2_output': 'done',
 'step3_output': 'done'}

In [11]:
workflow.get_state(config)

StateSnapshot(values={'input': 'start', 'step1_output': 'done', 'step2_output': 'done', 'step3_output': 'done'}, next=(), config={'configurable': {'thread_id': '42', 'checkpoint_ns': '', 'checkpoint_id': '1f0ae6cd-c3e0-6c3a-800d-58bb1e0073a7'}}, metadata={'source': 'loop', 'step': 13, 'parents': {}}, created_at='2025-10-21T10:58:22.323996+00:00', parent_config={'configurable': {'thread_id': '42', 'checkpoint_ns': '', 'checkpoint_id': '1f0ae6cd-c3db-6a0a-800c-95606da689bd'}}, tasks=(), interrupts=())

In [12]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1_output': 'done', 'step2_output': 'done', 'step3_output': 'done'}, next=(), config={'configurable': {'thread_id': '42', 'checkpoint_ns': '', 'checkpoint_id': '1f0ae6cd-c3e0-6c3a-800d-58bb1e0073a7'}}, metadata={'source': 'loop', 'step': 13, 'parents': {}}, created_at='2025-10-21T10:58:22.323996+00:00', parent_config={'configurable': {'thread_id': '42', 'checkpoint_ns': '', 'checkpoint_id': '1f0ae6cd-c3db-6a0a-800c-95606da689bd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1_output': 'done', 'step2_output': 'done', 'step3_output': 'done'}, next=('step3',), config={'configurable': {'thread_id': '42', 'checkpoint_ns': '', 'checkpoint_id': '1f0ae6cd-c3db-6a0a-800c-95606da689bd'}}, metadata={'source': 'loop', 'step': 12, 'parents': {}}, created_at='2025-10-21T10:58:22.321883+00:00', parent_config={'configurable': {'thread_id': '42', 'checkpoint_ns': '', 'checkpoint_id': '1f0ae6cc-a5b3-668c-800b-e859a588a50b'}}, task

## TimeTravel